# Simple Plan Node Prototype - Step 1: Registry from Sitemap

This notebook starts with the simplest possible registry builder:
1. Load `https://www.fretchen.eu/sitemap.xml`
2. Extract all `<loc>` URLs
3. Normalize URLs (remove query/fragment, stable trailing slash policy)
4. Save a basic registry JSON file

In [5]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
from urllib.request import urlopen
import xml.etree.ElementTree as ET

SITEMAP_URL = "https://www.fretchen.eu/sitemap.xml"
OUT_PATH = Path("../state/simple_planner/registry.json")


def normalize_url(url: str) -> str:
    parts = urlsplit(url.strip())
    path = parts.path or "/"
    if path != "/" and path.endswith("/"):
        path = path[:-1]
    return urlunsplit((parts.scheme.lower(), parts.netloc.lower(), path, "", ""))


with urlopen(SITEMAP_URL) as response:
    raw_xml = response.read()

root = ET.fromstring(raw_xml)
ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}
loc_nodes = root.findall("sm:url/sm:loc", ns)

registry_urls = []
seen = set()
for node in loc_nodes:
    normalized = normalize_url(node.text or "")
    if not normalized or normalized in seen:
        continue
    seen.add(normalized)
    registry_urls.append(normalized)

registry = {
    "source": SITEMAP_URL,
    "count": len(registry_urls),
    "urls": registry_urls,
}

print(f"Loaded {registry['count']} URLs from sitemap")
registry_urls[:10]

Loaded 73 URLs from sitemap


['https://www.fretchen.eu/',
 'https://www.fretchen.eu/blog',
 'https://www.fretchen.eu/quantum',
 'https://www.fretchen.eu/blog/0',
 'https://www.fretchen.eu/blog/1',
 'https://www.fretchen.eu/blog/10',
 'https://www.fretchen.eu/blog/11',
 'https://www.fretchen.eu/blog/12',
 'https://www.fretchen.eu/blog/13',
 'https://www.fretchen.eu/blog/14']

In [6]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(json.dumps(registry, indent=2), encoding="utf-8")
print(f"Wrote registry to {OUT_PATH.resolve()}")

# quick sanity check
print("First 5 registry URLs:")
for url in registry['urls'][:5]:
    print("-", url)

Wrote registry to /Users/fredjendrzejewski/GitHub/fretchen.github.io/growth-agent/state/simple_planner/registry.json
First 5 registry URLs:
- https://www.fretchen.eu/
- https://www.fretchen.eu/blog
- https://www.fretchen.eu/quantum
- https://www.fretchen.eu/blog/0
- https://www.fretchen.eu/blog/1


Writing the registry for the moment locally for some first tests. However, later it shall be saved with the other files in the S3 bucket.

In [7]:
# Build a cleaned registry by removing hand-curated excluded URLs.
# Expected structure in registry.json:
# {
#   "source": "...",
#   "count": 123,
#   "urls": [...],
#   "excluded": [...]
# }

CLEAN_OUT_PATH = OUT_PATH.with_name("registry_clean.json")

raw_registry = json.loads(OUT_PATH.read_text(encoding="utf-8"))
excluded_raw = raw_registry.get("excluded", [])

excluded_normalized = {normalize_url(u) for u in excluded_raw if isinstance(u, str) and u.strip()}

clean_urls = []
for u in raw_registry.get("urls", []):
    nu = normalize_url(u)
    if nu in excluded_normalized:
        continue
    clean_urls.append(nu)

clean_registry = {
    "source": raw_registry.get("source", SITEMAP_URL),
    "count": len(clean_urls),
    "urls": clean_urls,
    "excluded": sorted(excluded_normalized),
}

CLEAN_OUT_PATH.write_text(json.dumps(clean_registry, indent=2), encoding="utf-8")

print(f"Excluded: {len(excluded_normalized)}")
print(f"Original URLs: {len(raw_registry.get('urls', []))}")
print(f"Clean URLs: {clean_registry['count']}")
print(f"Wrote cleaned registry to {CLEAN_OUT_PATH.resolve()}")

Excluded: 0
Original URLs: 73
Clean URLs: 73
Wrote cleaned registry to /Users/fredjendrzejewski/GitHub/fretchen.github.io/growth-agent/state/simple_planner/registry_clean.json


In [8]:
# Read the real queue from S3 and build "latest publication per page" history.
# Uses the same S3 location as scw_js/growth_service.ts.

import os
from datetime import datetime, timezone
from pathlib import Path

import boto3
from dotenv import load_dotenv

S3_BUCKET = "my-imagestore"
S3_REGION = "nl-ams"
S3_ENDPOINT = "https://s3.nl-ams.scw.cloud"
S3_STATE_KEY = "growth-agent/content_queue.json"

# Load environment variables from growth-agent/.env (notebooks run from notebooks/)
dotenv_candidates = [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
]
for dotenv_path in dotenv_candidates:
    if dotenv_path.exists():
        load_dotenv(dotenv_path=dotenv_path, override=False)
        print(f"Loaded env from {dotenv_path}")
        break

access_key = os.environ.get("SCW_ACCESS_KEY")
secret_key = os.environ.get("SCW_SECRET_KEY")
if not access_key or not secret_key:
    raise RuntimeError(
        "Missing SCW_ACCESS_KEY/SCW_SECRET_KEY. Add them to growth-agent/.env or export them in shell."
    )

s3 = boto3.client(
    "s3",
    region_name=S3_REGION,
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

obj = s3.get_object(Bucket=S3_BUCKET, Key=S3_STATE_KEY)
queue = json.loads(obj["Body"].read())

published = queue.get("published", [])
now = datetime.now(timezone.utc)

latest_by_url = {}
for draft in published:
    link = draft.get("link")
    if not link:
        continue

    page_url = normalize_url(link)

    ts_raw = draft.get("scheduled_at") or draft.get("created")
    if not ts_raw:
        continue

    ts = datetime.fromisoformat(ts_raw.replace("Z", "+00:00"))
    if page_url not in latest_by_url or ts > latest_by_url[page_url]:
        latest_by_url[page_url] = ts

publication_history = []
for page_url, last_ts in latest_by_url.items():
    t_days = (now - last_ts).total_seconds() / 86400.0
    publication_history.append(
        {
            "page_url": page_url,
            "last_published_at": last_ts.isoformat(),
            "t_days": round(t_days, 2),
        }
    )

publication_history.sort(key=lambda x: x["t_days"])

print(f"Loaded queue from s3://{S3_BUCKET}/{S3_STATE_KEY}")
print(f"Published drafts in queue: {len(published)}")
print(f"Unique pages with publication history: {len(publication_history)}")
publication_history[:10]

Loaded env from /Users/fredjendrzejewski/GitHub/fretchen.github.io/growth-agent/.env
Loaded queue from s3://my-imagestore/growth-agent/content_queue.json
Published drafts in queue: 8
Unique pages with publication history: 3


[{'page_url': 'https://www.fretchen.eu/x402',
  'last_published_at': '2026-04-22T07:00:00+00:00',
  't_days': 1.58},
 {'page_url': 'https://www.fretchen.eu/quantum/amo',
  'last_published_at': '2026-04-20T09:00:00+00:00',
  't_days': 3.5},
 {'page_url': 'https://www.fretchen.eu/blog/25',
  'last_published_at': '2026-04-18T09:00:00+00:00',
  't_days': 5.5}]

In [ ]:
# Draw next pages using half-life decay on recently published pages.
# KISS approach: start from clean registry, reduce weight by recency only.

import random
import math
from urllib.parse import urlsplit, urlunsplit

N_SAMPLES = 3
HALF_LIFE_DAYS = 30.0
RANDOM_SEED = 43

# random.seed(RANDOM_SEED)


def canonical_page_url(url: str) -> str:
    p = urlsplit((url or "").strip())
    path = p.path or "/"
    if path != "/" and path.endswith("/"):
        path = path[:-1]
    return urlunsplit((p.scheme.lower(), p.netloc.lower(), path, "", ""))


# Build quick lookup: page -> days since last publication
last_days_by_page = {
    canonical_page_url(item["page_url"]): float(item["t_days"])
    for item in publication_history
}

candidate_urls = [canonical_page_url(u) for u in clean_registry.get("urls", [])]

weighted_candidates = []
for page_url in candidate_urls:
    t_days = last_days_by_page.get(page_url)

    if t_days is None:
        weight = 1.0
    else:
        # Pure half-life recovery: recently used pages get lower weight,
        # then recover smoothly over time. Clamp to zero for robustness.
        weight = max(0.0, 1.0 - math.pow(2.0, -t_days / HALF_LIFE_DAYS))

    weighted_candidates.append(
        {
            "page_url": page_url,
            "t_days": t_days,
            "weight": weight,
        }
    )

eligible = [c for c in weighted_candidates if c["weight"] > 0]
if not eligible:
    raise RuntimeError("No eligible pages to sample. Check timestamps/history.")

# Weighted sample without replacement (simple iterative KISS)
chosen = []
pool = eligible.copy()
for _ in range(min(N_SAMPLES, len(pool))):
    weights = [c["weight"] for c in pool]
    picked = random.choices(pool, weights=weights, k=1)[0]
    chosen.append(picked)
    pool = [c for c in pool if c["page_url"] != picked["page_url"]]

print(f"Candidates total: {len(candidate_urls)}")
print(f"Eligible for draw: {len(eligible)}")
print(f"Sampled pages: {len(chosen)}")
print("\nSelected pages:")
for i, item in enumerate(chosen, start=1):
    print(f"{i}. {item['page_url']} | t_days={item['t_days']} | weight={item['weight']:.4f}")

chosen

https://www.fretchen.eu/ | t_days=None | weight=1.0000
https://www.fretchen.eu/blog | t_days=None | weight=1.0000
https://www.fretchen.eu/quantum | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/0 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/1 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/10 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/11 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/12 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/13 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/14 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/15 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/16 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/17 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/18 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/19 | t_days=None | weight=1.0000
https://www.fretchen.eu/blog/2 | t_days=None | weight=1.0000
https://www.fretchen.

[{'page_url': 'https://www.fretchen.eu/quantum/amo/10',
  't_days': None,
  'weight': 1.0},
 {'page_url': 'https://www.fretchen.eu/quantum/amo/14',
  't_days': None,
  'weight': 1.0},
 {'page_url': 'https://www.fretchen.eu/blog', 't_days': None, 'weight': 1.0}]